# Chart Patterns

## Contents

- [Download Data](#download-data)
- [Visualization](#visualization)
- [Support and Resistance](#1-support-and-resistance)
- [Trend Line](#2-trend-line)
- [Double Top](#3-double-top)
- [False Breakout](#4-false-breakout)

## Download Data
[top](#contents)

This section provides functions to download and prepare S&P 500 (SPX) market data for chart pattern analysis.

### Key Functions:
- **`get_start_and_end(days_back=365)`**: Generates date ranges for data fetching
- **`get_spx_data(start, end)`**: Downloads SPX data from Yahoo Finance and prepares it

### Data Preparation Process:
1. **Date Range Generation**: Creates start and end dates based on lookback period
2. **Data Download**: Fetches OHLC data using yfinance
3. **Data Cleaning**: Flattens multi-index columns, rounds prices to 2 decimals
4. **Local Storage**: Saves data to CSV for future use

### Usage Example:
```python
# Generate dates for last 365 days
start, end = get_start_and_end(days_back=365)

# Download SPX data
df = get_spx_data(start, end)

# Data is now ready for pattern analysis
```

The resulting DataFrame contains:
- **Date Index**: Trading dates
- **OHLC Columns**: Open, High, Low, Close prices
- **Volume**: Trading volume data

In [ ]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime, timedelta


def get_start_and_end(days_back=365):
    """
    Generate start and end dates for data fetching
    
    Args:
        days_back (int): Number of days to look back from today
        
    Returns:
        tuple: (start_date, end_date) as strings in 'YYYY-MM-DD' format
    """
    # Get current date as end point
    end_date = datetime.now()
    
    # Calculate start date by subtracting specified days
    start_date = end_date - timedelta(days=days_back)
    
    # Return formatted date strings
    return start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')


def get_spx_data(start: str, end: str) -> pd.DataFrame:
    """
    Download S&P 500 historical data from Yahoo Finance
    
    Args:
        start (str): Start date in 'YYYY-MM-DD' format
        end (str): End date in 'YYYY-MM-DD' format
        
    Returns:
        pd.DataFrame: OHLCV data with Date index
    """
    # Define the ticker symbol for the S&P 500 index
    ticker = "^GSPC"  # Yahoo Finance symbol for S&P 500

    # Download historical data using yfinance
    # auto_adjust=True applies dividend and stock split adjustments
    df = yf.download(ticker, start=start, end=end, interval="1d", auto_adjust=True)

    # Handle multi-index columns that yfinance sometimes returns
    if isinstance(df.columns, pd.MultiIndex):
        # Flatten the multi-index structure
        column_names = ["Date"] + df.columns.get_level_values(0).tolist()
        df = df.reset_index()
        df.columns = column_names
        df = df.set_index("Date")

    # Round prices to 2 decimal places for cleaner display
    # This also helps with floating point precision issues
    df[["Open", "High", "Low", "Close"]] = df[["Open", "High", "Low", "Close"]].round(2)

    # Create data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    
    # Save to CSV in local data directory for future use
    df.to_csv("data/spx.csv")
    print("Data saved to data/spx.csv")

    return df

In [ ]:
# Generate start and end dates (default 1 year back)
start, end = get_start_and_end(days_back=365)
print(f"Fetching SPX data from {start} to {end}")

# Fetch SPX data using local function
df = get_spx_data(start, end)

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())

## Visualization
[top](#contents)

This section provides OHLC (Open, High, Low, Close) chart visualization functions using **mplfinance** for professional trading-style charts.

### Key Features:
- **Candlestick Charts**: Professional OHLC visualization with proper candlestick styling
- **Pattern Overlays**: Support for adding trend lines, support/resistance levels, and pattern markers
- **Volume Display**: Optional volume subplot for comprehensive analysis
- **Interactive Elements**: Annotations and markers for detected patterns

### Chart Types Available:
- **Candlestick**: Traditional Japanese candlestick charts (default)
- **OHLC Bars**: Traditional OHLC bar charts
- **Line**: Simple line chart overlay option

### Benefits over Line Charts:
- **Price Action Visibility**: Shows open, high, low, close for each period
- **Market Psychology**: Candlestick patterns reveal buying/selling pressure
- **Pattern Recognition**: Better visualization of support/resistance tests and breakouts
- **Professional Appearance**: Industry-standard chart format

In [ ]:
import matplotlib.pyplot as plt
import mplfinance as mpf
import numpy as np


def create_ohlc_chart(data, title="Price Chart", volume=True, style='charles', figsize=(15, 10)):
    """
    Create a basic OHLC candlestick chart using mplfinance
    
    Args:
        data (pd.DataFrame): DataFrame with OHLC data (Open, High, Low, Close, Volume)
        title (str): Chart title
        volume (bool): Whether to show volume subplot
        style (str): Chart style ('charles', 'yahoo', 'nightclouds', etc.)
        figsize (tuple): Figure size (width, height)
        
    Returns:
        tuple: (fig, axes) matplotlib figure and axes objects
    """
    # Ensure data has required columns
    required_cols = ['Open', 'High', 'Low', 'Close']
    if not all(col in data.columns for col in required_cols):
        raise ValueError(f"Data must contain columns: {required_cols}")
    
    # Create the chart
    fig, axes = mpf.plot(
        data,
        type='candle',           # Candlestick chart
        style=style,             # Chart style
        volume=volume,           # Show volume
        figsize=figsize,         # Chart size
        title=title,             # Chart title
        returnfig=True,          # Return figure object
        tight_layout=True        # Optimize layout
    )
    
    return fig, axes


def plot_support_resistance_ohlc(data, support_resistance, title="Support & Resistance Analysis", 
                                volume=True, figsize=(15, 10)):
    """
    Plot OHLC chart with support and resistance levels using mplfinance
    
    Args:
        data (pd.DataFrame): DataFrame with OHLC data
        support_resistance (dict): Dictionary with 'support' and 'resistance' lists containing (date, price) tuples
        title (str): Chart title
        volume (bool): Whether to show volume subplot
        figsize (tuple): Figure size
    """
    # Create horizontal lines for support and resistance levels
    addplots = []
    
    # Add support levels (green dashed lines) - now handling (date, price) tuples
    for i, (date, level) in enumerate(support_resistance["support"]):
        # Create a horizontal line across the entire data range
        support_line = pd.Series([level] * len(data), index=data.index, name=f'Support_{i+1}')
        addplots.append(
            mpf.make_addplot(support_line, color='green', linestyle='--', width=2, alpha=0.7)
        )
    
    # Add resistance levels (red dashed lines) - now handling (date, price) tuples
    for i, (date, level) in enumerate(support_resistance["resistance"]):
        # Create a horizontal line across the entire data range
        resistance_line = pd.Series([level] * len(data), index=data.index, name=f'Resistance_{i+1}')
        addplots.append(
            mpf.make_addplot(resistance_line, color='red', linestyle='--', width=2, alpha=0.7)
        )
    
    # Create the chart with addplots
    fig, axes = mpf.plot(
        data,
        type='candle',
        style='charles',
        volume=volume,
        addplot=addplots,
        figsize=figsize,
        title=title,
        returnfig=True,
        tight_layout=True
    )
    
    # Add legend manually since mplfinance doesn't auto-generate legends for addplots
    if len(support_resistance["support"]) > 0 or len(support_resistance["resistance"]) > 0:
        legend_elements = []
        
        # Add support legend entries with dates
        for i, (date, level) in enumerate(support_resistance["support"]):
            date_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)
            legend_elements.append(
                plt.Line2D([0], [0], color='green', linestyle='--', 
                          label=f'Support: ${level:.2f} ({date_str})', linewidth=2)
            )
        
        # Add resistance legend entries with dates
        for i, (date, level) in enumerate(support_resistance["resistance"]):
            date_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)
            legend_elements.append(
                plt.Line2D([0], [0], color='red', linestyle='--', 
                          label=f'Resistance: ${level:.2f} ({date_str})', linewidth=2)
            )
        
        # Add legend to the main price chart (first axis)
        axes[0].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1))
    
    plt.tight_layout()
    return fig, axes


# Example usage functions
def demo_basic_ohlc_chart(data):
    """Demo function to show basic OHLC chart"""
    print("Creating basic OHLC candlestick chart...")
    fig, axes = create_ohlc_chart(data, title="S&P 500 OHLC Chart")
    plt.show()
    return fig, axes


def demo_support_resistance_ohlc(data, levels):
    """Demo function to show S&R levels on OHLC chart"""
    print("Creating OHLC chart with support and resistance levels...")
    fig, axes = plot_support_resistance_ohlc(data, levels, 
                                           title="S&P 500 with Support & Resistance")
    plt.show()
    return fig, axes

In [ ]:
# Demo the new OHLC chart with support and resistance levels
try:
    # First, let's show a basic OHLC chart
    print("=== BASIC OHLC CHART DEMO ===")
    demo_basic_ohlc_chart(df)  
except Exception as e:
    print(f"Error creating OHLC charts: {e}")

## 1. Support and Resistance Detection
[top](#contents)

Support and resistance levels are **horizontal price levels** where the price tends to reverse or consolidate. These levels represent psychological barriers in the market:

- **Support**: A price level where buying interest prevents further decline
- **Resistance**: A price level where selling pressure prevents further rise

### Algorithm Logic:

1. **Peak Detection**: Use `scipy.signal.argrelextrema` to find local minima (support) and maxima (resistance)
2. **Level Clustering**: Group nearby price levels within a threshold percentage
3. **Validation**: Filter levels that have been "touched" by price multiple times
4. **Significance**: Only keep levels with minimum number of touches (e.g., 3+)

### Key Parameters:
- **`window`**: Lookback period for peak detection (default: 20 days)
- **`min_touches`**: Minimum touches required to confirm a level (default: 3)
- **`threshold`**: Price tolerance for level clustering (default: 2%)

### Mathematical Approach:
```python
# Find peaks and troughs
minima = argrelextrema(close_prices, np.less, order=window)
maxima = argrelextrema(close_prices, np.greater, order=window)

# Cluster similar levels
if abs(price1 - price2) / price1 <= threshold:
    # Prices are considered same level
    
# Count touches
touches = sum(abs(close_price - level) / level <= threshold)
```

### Output:
Returns a dictionary containing:
- **`support`**: List of significant support levels
- **`resistance`**: List of significant resistance levels

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema


def cluster_levels(date_value_pairs, threshold):
        """
        Cluster nearby price levels within threshold percentage, preserving earliest date

        Args:
            date_value_pairs (list): List of (date, value) tuples to cluster
            threshold (float): Clustering threshold as percentage

        Returns:
            list: List of (date, clustered_value) tuples with earliest date in cluster
        """
        if not date_value_pairs:
            return []

        clustered = []
        # Sort by price value for sequential processing
        sorted_pairs = sorted(date_value_pairs, key=lambda x: x[1])
        i = 0

        while i < len(sorted_pairs):
            # Start new cluster with current level
            cluster_pairs = [sorted_pairs[i]]
            j = i + 1

            # Add nearby levels to same cluster
            while (j < len(sorted_pairs) and
                   abs(sorted_pairs[j][1] - sorted_pairs[i][1]) / sorted_pairs[i][1] <= threshold):
                cluster_pairs.append(sorted_pairs[j])
                j += 1

            # Find earliest date in cluster and calculate average price
            earliest_date = min(cluster_pairs, key=lambda x: x[0])[0]
            avg_price = np.mean([pair[1] for pair in cluster_pairs])

            clustered.append((earliest_date, avg_price))
            i = j  # Move to next unprocessed level

        return clustered


def find_support_resistance(data, window=20, min_touches=3, threshold=0.02):
    """
    Detect support and resistance levels using peak detection and clustering.

    Args:
        data (pd.DataFrame): DataFrame with 'Close' price column and datetime index
        window (int): Lookback window for finding peaks/troughs
        min_touches (int): Minimum number of touches to confirm a level
        threshold (float): Price range to consider a level (as a percentage)

    Returns:
        dict: Dictionary with 'support' and 'resistance' lists containing (date, price) tuples
    """
    # Step 1: Find local minima (potential support) and maxima (potential resistance)
    # argrelextrema finds indices where values are extrema relative to neighbors
    minima_idx = argrelextrema(data["Close"].values, np.less, order=window)[0]
    maxima_idx = argrelextrema(data["Close"].values, np.greater, order=window)[0]

    # Step 2: Extract date-price pairs at these extrema
    support_date_values = [(data.index[i], data["Close"].iloc[i]) for i in minima_idx]
    resistance_date_values = [(data.index[i], data["Close"].iloc[i]) for i in maxima_idx]

    # Step 3: Filter levels with sufficient touches (validation)
    supports = []
    resistances = []

    # Process support levels
    for date, level in cluster_levels(support_date_values, threshold):
        # Count how many times price touched this level
        # A "touch" is when price comes within threshold percentage of level
        touches = sum(abs(data["Close"] - level) / level <= threshold)

        # Only keep levels with enough touches for significance
        if touches >= min_touches:
            supports.append((date, level))

    # Process resistance levels
    for date, level in cluster_levels(resistance_date_values, threshold):
        # Count touches for resistance levels
        touches = sum(abs(data["Close"] - level) / level <= threshold)

        # Validate resistance level significance
        if touches >= min_touches:
            resistances.append((date, level))

    return {"support": supports, "resistance": resistances}

In [ ]:
# Example usage with data loading and analysis
try:
    # Load data from CSV if available, otherwise use the DataFrame from previous cells
    if 'df' not in locals():
        df = pd.read_csv("data/spx.csv", index_col=0, parse_dates=True)
    
    # Detect support and resistance levels (now returns date-price tuples)
    levels = find_support_resistance(df, window=20, min_touches=2, threshold=0.03)
    
    # Display results
    print("=== ENHANCED SUPPORT AND RESISTANCE ANALYSIS ===" )
    print(f"Analysis period: {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
    print(f"Total data points: {len(df)}")
    print()
    
    print(f"Support Levels Found: {len(levels['support'])}")
    for i, (date, level) in enumerate(levels['support'], 1):
        date_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)
        print(f"  {i}. ${level:.2f} (first occurred: {date_str})")
    
    print(f"\nResistance Levels Found: {len(levels['resistance'])}")
    for i, (date, level) in enumerate(levels['resistance'], 1):
        date_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)
        print(f"  {i}. ${level:.2f} (first occurred: {date_str})")
        
    # Show current price for context
    current_price = df['Close'].iloc[-1]
    print(f"\nCurrent Price: ${current_price:.2f}")
    print()
    
    # Analyze level significance relative to current price
    print("=== LEVEL ANALYSIS ===")
    for i, (date, level) in enumerate(levels['support'], 1):
        distance_pct = ((current_price - level) / level) * 100
        print(f"Support {i}: ${level:.2f} is {distance_pct:.1f}% below current price")
    
    for i, (date, level) in enumerate(levels['resistance'], 1):
        distance_pct = ((level - current_price) / current_price) * 100
        print(f"Resistance {i}: ${level:.2f} is {distance_pct:.1f}% above current price")
        
except Exception as e:
    print(f"Error in support/resistance analysis: {e}")
    print("Make sure data is loaded properly")

In [ ]:
# Demo the new OHLC chart with support and resistance levels
try:
    print("\n=== SUPPORT & RESISTANCE ON OHLC CHART ===")
    demo_support_resistance_ohlc(df, levels)
except Exception as e:
    print(f"Error creating OHLC charts: {e}")
    print("Showing basic OHLC chart as fallback...")
    demo_basic_ohlc_chart(df)

## 2. Trend Line Detection
[top](#contents)

A **trend line** is a straight line connecting consecutive pivot points that shows the general direction of price movement:

- **Uptrend Line**: Connects successive higher lows (rising support)
- **Downtrend Line**: Connects successive lower highs (declining resistance)

### Algorithm Logic:

1. **Pivot Point Detection**: Find local highs and lows using peak detection
2. **Linear Regression**: Fit a straight line through pivot points using least squares
3. **Trend Validation**: Check if enough price points stay near the trend line
4. **Slope Validation**: Ensure uptrends have positive slopes, downtrends negative

### Key Parameters:
- **`window`**: Lookback period for pivot detection (default: 20 days)
- **`min_touches`**: Minimum price points near trend line (default: 3)
- **`threshold`**: Maximum deviation from trend line (default: 2%)

### Mathematical Foundation:
```python
# Linear regression for trend line
slope, intercept = linregress(x_indices, y_prices)
trend_line = slope * x + intercept

# Validation: count points within threshold
deviation = abs(actual_price - trend_price) / actual_price
valid_if: deviation <= threshold
```

### Trend Line Types:
- **Uptrend**: Slope > 0, connects higher lows
- **Downtrend**: Slope < 0, connects lower highs

### Quality Metrics:
- **R-squared**: Measures how well the line fits the data
- **Touch Count**: Number of price points near the line
- **Slope**: Steepness of the trend (positive/negative)

In [ ]:
from scipy.stats import linregress


def detect_trend_line(data, window=20, min_touches=3, threshold=0.02):
    """
    Detect trend lines (uptrend or downtrend) using linear regression on pivot points.
    
    Args:
        data (pd.DataFrame): DataFrame with 'Close' price column
        window (int): Lookback window for finding pivots
        min_touches (int): Minimum number of touches to confirm trend line
        threshold (float): Acceptable deviation from trend line (as a percentage)
        
    Returns:
        list: List of dictionaries containing trend line parameters
    """
    # Step 1: Find pivot points using peak detection
    # For uptrends, we need lows (potential support points)
    # For downtrends, we need highs (potential resistance points)
    lows_idx = argrelextrema(data["Close"].values, np.less, order=window)[0]
    highs_idx = argrelextrema(data["Close"].values, np.greater, order=window)[0]

    def fit_trend_line(indices):
        """
        Fit a linear regression line through pivot points
        
        Args:
            indices (array): Array of pivot point indices
            
        Returns:
            tuple: (slope, intercept, r_squared) or None if insufficient data
        """
        # Need at least 2 points to fit a line
        if len(indices) < 2:
            return None
            
        # Use indices as x-coordinates, prices as y-coordinates
        x = indices
        y = data["Close"].iloc[indices].values
        
        # Perform linear regression to find best-fit line
        slope, intercept, r_value, _, _ = linregress(x, y)
        
        # Return slope, intercept, and quality measure (r-squared)
        return slope, intercept, r_value**2

    def validate_trend(slope, intercept, threshold, min_touches):
        """
        Validate if enough price points stay close to the trend line
        
        Args:
            slope (float): Trend line slope
            intercept (float): Trend line y-intercept  
            threshold (float): Maximum allowed deviation
            min_touches (int): Minimum required touches
            
        Returns:
            tuple: (slope, intercept, touches) or None if invalid
        """
        # Skip if we don't have valid parameters
        if slope is None or intercept is None:
            return None
        
        # Calculate trend line values for all data points
        # Formula: y = mx + b (slope * x + intercept)
        x = np.arange(len(data))
        trend_line = slope * x + intercept
        
        # Count how many price points are "close" to the trend line
        # "Close" means within threshold percentage of the trend line price
        relative_deviation = abs(data["Close"] - trend_line) / data["Close"]
        touches = sum(relative_deviation <= threshold)
        
        # Only accept trend if it has enough touches for significance
        if touches >= min_touches:
            return slope, intercept, touches
        return None

    result = []
    
    # Step 2: Check for uptrend (connecting higher lows)
    if len(lows_idx) >= 2:  # Need at least 2 lows for a trend line
        uptrend = fit_trend_line(lows_idx)
        
        if uptrend:
            # Validate the uptrend line
            uptrend_result = validate_trend(uptrend[0], uptrend[1], threshold, min_touches)
            
            # For uptrends, slope must be positive (rising)
            if uptrend_result and uptrend_result[0] > 0:
                result.append({
                    "type": "uptrend",
                    "slope": uptrend_result[0],           # Rate of price increase
                    "intercept": uptrend_result[1],       # Starting price level
                    "touches": uptrend_result[2],         # Validation strength
                    "r_squared": uptrend[2],              # Line fit quality
                    "pivot_points": len(lows_idx)         # Number of lows used
                })
    
    # Step 3: Check for downtrend (connecting lower highs)
    if len(highs_idx) >= 2:  # Need at least 2 highs for a trend line
        downtrend = fit_trend_line(highs_idx)
        
        if downtrend:
            # Validate the downtrend line
            downtrend_result = validate_trend(downtrend[0], downtrend[1], threshold, min_touches)
            
            # For downtrends, slope must be negative (falling)
            if downtrend_result and downtrend_result[0] < 0:
                result.append({
                    "type": "downtrend", 
                    "slope": downtrend_result[0],         # Rate of price decrease
                    "intercept": downtrend_result[1],     # Starting price level
                    "touches": downtrend_result[2],       # Validation strength
                    "r_squared": downtrend[2],            # Line fit quality
                    "pivot_points": len(highs_idx)        # Number of highs used
                })

    return result

In [ ]:
def plot_trend_lines_ohlc(data, trends, title="Trend Line Analysis", volume=True, 
                         show_pivots=True, figsize=(15, 10)):
    """
    Plot OHLC chart with detected trend lines using mplfinance
    
    Args:
        data (pd.DataFrame): DataFrame with OHLC data
        trends (list): List of trend dictionaries from detect_trend_line
        title (str): Chart title
        volume (bool): Whether to show volume subplot
        show_pivots (bool): Whether to show pivot points on chart
        figsize (tuple): Figure size
    """
    addplots = []
    
    # Create trend lines as addplots
    for i, trend in enumerate(trends):
        # Calculate trend line values using y = mx + b formula
        x = np.arange(len(data))
        trend_line_values = trend['slope'] * x + trend['intercept']
        
        # Create pandas Series with same index as data
        trend_series = pd.Series(trend_line_values, index=data.index, 
                               name=f"{trend['type']}_{i+1}")
        
        # Color based on trend type
        color = 'green' if trend['type'] == 'uptrend' else 'red'
        
        # Add trend line to addplots
        addplots.append(
            mpf.make_addplot(trend_series, color=color, linestyle='--', 
                           width=2.5, alpha=0.8)
        )
    
    # Add pivot points if requested
    if show_pivots:
        # Find pivot lows (for uptrend analysis)
        lows_idx = argrelextrema(data["Close"].values, np.less, order=15)[0]
        if len(lows_idx) > 0:
            # Create scatter plot for pivot lows
            pivot_lows = pd.Series([np.nan] * len(data), index=data.index)
            pivot_lows.iloc[lows_idx] = data["Close"].iloc[lows_idx]
            addplots.append(
                mpf.make_addplot(pivot_lows, type='scatter', markersize=100, 
                               marker='^', color='green', alpha=0.7)
            )
        
        # Find pivot highs (for downtrend analysis)
        highs_idx = argrelextrema(data["Close"].values, np.greater, order=15)[0]
        if len(highs_idx) > 0:
            # Create scatter plot for pivot highs
            pivot_highs = pd.Series([np.nan] * len(data), index=data.index)
            pivot_highs.iloc[highs_idx] = data["Close"].iloc[highs_idx]
            addplots.append(
                mpf.make_addplot(pivot_highs, type='scatter', markersize=100, 
                               marker='v', color='red', alpha=0.7)
            )
    
    # Create the chart with addplots
    fig, axes = mpf.plot(
        data,
        type='candle',
        style='charles',
        volume=volume,
        addplot=addplots if addplots else None,
        figsize=figsize,
        title=title,
        returnfig=True,
        tight_layout=True
    )
    
    # Create custom legend
    if trends or show_pivots:
        legend_elements = []
        
        # Add trend line legend entries
        for i, trend in enumerate(trends):
            color = 'green' if trend['type'] == 'uptrend' else 'red'
            legend_elements.append(
                plt.Line2D([0], [0], color=color, linestyle='--', 
                          label=f"{trend['type'].capitalize()} (R²={trend['r_squared']:.3f})", 
                          linewidth=2.5)
            )
        
        # Add pivot point legend entries
        if show_pivots:
            if len(argrelextrema(data["Close"].values, np.less, order=15)[0]) > 0:
                legend_elements.append(
                    plt.Line2D([0], [0], marker='^', color='green', linestyle='', 
                              markersize=8, label='Pivot Lows', alpha=0.7)
                )
            if len(argrelextrema(data["Close"].values, np.greater, order=15)[0]) > 0:
                legend_elements.append(
                    plt.Line2D([0], [0], marker='v', color='red', linestyle='', 
                              markersize=8, label='Pivot Highs', alpha=0.7)
                )
        
        # Add legend to the main price chart (first axis)
        if legend_elements:
            axes[0].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1))
    
    plt.tight_layout()
    return fig, axes


def demo_trend_lines_ohlc(data, trends):
    """Demo function to show trend lines on OHLC chart"""
    print("Creating OHLC chart with trend lines...")
    fig, axes = plot_trend_lines_ohlc(data, trends, 
                                    title="S&P 500 Trend Line Analysis")
    plt.show()
    return fig, axes

In [ ]:
# Example usage - Comprehensive Trend Line Analysis
try:
    print("=== TREND LINE DETECTION ANALYSIS ===")
    print("Detecting trend lines in price data...")
    
    # Run trend detection with moderate sensitivity
    trends = detect_trend_line(df, window=15, min_touches=2, threshold=0.03)
    
    if trends:
        print(f"\nDetected {len(trends)} trend line(s):")
        print("-" * 50)
        
        for i, trend in enumerate(trends, 1):
            print(f"{i}. {trend['type'].upper()}:")
            print(f"   • Slope: {trend['slope']:.6f} (price change per day)")
            print(f"   • Intercept: {trend['intercept']:.2f} (starting price level)")
            print(f"   • Touches: {trend['touches']} (validation strength)")
            print(f"   • R-squared: {trend['r_squared']:.4f} (line fit quality)")
            print(f"   • Pivot points: {trend['pivot_points']} (data points used)")
            
            # Interpret the slope
            if trend['type'] == 'uptrend':
                daily_increase = trend['slope']
                print(f"   • Daily increase: ~${daily_increase:.2f} per day")
            else:
                daily_decrease = abs(trend['slope'])
                print(f"   • Daily decrease: ~${daily_decrease:.2f} per day")
            print()
        
        # Plot the results with OHLC charts
        print("Plotting trend lines on OHLC chart...")
        demo_trend_lines_ohlc(df, trends)
        
    else:
        print("No significant trend lines detected with current parameters")
        print("Try adjusting window, min_touches, or threshold parameters")
        print("Showing basic OHLC chart instead...")
        demo_basic_ohlc_chart(df)
        
        print("\nTips to find trend lines:")
        print("• Try reducing 'window' for more sensitive pivot detection")
        print("• Try reducing 'min_touches' for less strict validation") 
        print("• Try increasing 'threshold' for more flexible line fitting")
        
except Exception as e:
    print(f"Error in trend line analysis: {e}")
    print("Make sure 'df' variable exists and contains OHLC data")

## 3. Double Top Pattern Detection
[top](#contents)

A **double top** is a **bearish reversal pattern** that signals a potential trend change from upward to downward movement:

### Pattern Structure:
1. **First Peak**: Price reaches a high point
2. **Decline**: Price pulls back to form a valley
3. **Second Peak**: Price rallies back to similar high level
4. **Neckline**: The low point between the two peaks
5. **Breakdown**: Price falls below the neckline, confirming the pattern

### Visual Representation:
```
    Peak1      Peak2
      /\        /\
     /  \      /  \
    /    \    /    \
   /      \  /      \
  /        \/        \
              Neckline ----------------
                      \
                       \  Breakdown
                        \/
```

### Algorithm Logic:

1. **Peak Detection**: Find local maxima using `argrelextrema`
2. **Peak Comparison**: Check if consecutive peaks are similar in price
3. **Neckline Identification**: Find the lowest low between peaks
4. **Breakdown Confirmation**: Verify price breaks below neckline after second peak

### Key Parameters:
- **`window`**: Lookback period for peak detection (default: 20 days)
- **`threshold`**: Maximum price difference between peaks (default: 2%)

### Trading Implications:
- **Bearish Signal**: Suggests uptrend exhaustion
- **Price Target**: Typically neckline - (peak_height - neckline)
- **Stop Loss**: Usually placed above the second peak

In [ ]:
def detect_double_top(data, window=20, threshold=0.02):
    """
    Detect double top pattern.
    :param data: DataFrame with 'Close' price column
    :param window: Lookback window for finding peaks
    :param threshold: Price range to consider peaks as similar
    :return: List of detected double top patterns
    """
    maxima_idx = argrelextrema(data["Close"].values, np.greater, order=window)[0]
    patterns = []

    for i in range(len(maxima_idx) - 1):
        peak1 = data["Close"].iloc[maxima_idx[i]]
        peak2 = data["Close"].iloc[maxima_idx[i + 1]]
        if abs(peak1 - peak2) / peak1 <= threshold:
            # Find neckline (lowest low between peaks)
            neckline_idx = data["Low"].iloc[maxima_idx[i] : maxima_idx[i + 1]].idxmin()
            neckline = data["Low"].loc[neckline_idx]
            
            # Check for breakdown below neckline after the second peak
            # Get all data points after the second peak
            second_peak_idx = maxima_idx[i + 1]
            remaining_data = data.iloc[second_peak_idx + 1:]
            
            if len(remaining_data) > 0:
                # Check if any subsequent close price breaks below neckline
                breakdown = remaining_data["Close"].lt(neckline).any()
                if breakdown:
                    # Find the first breakdown point (relative to original data)
                    breakdown_series = remaining_data["Close"].lt(neckline)
                    first_breakdown_loc = breakdown_series.idxmax() if breakdown_series.any() else None
                    
                    if first_breakdown_loc is not None:
                        # Convert the timestamp back to integer index position
                        breakdown_point = data.index.get_loc(first_breakdown_loc)
                    else:
                        breakdown_point = second_peak_idx + 1  # Default to next candle
                    
                    patterns.append({
                        "peak1_idx": maxima_idx[i],
                        "peak2_idx": maxima_idx[i + 1],
                        "neckline": neckline,
                        "breakdown_idx": breakdown_point,
                    })

    return patterns

In [ ]:
def plot_double_top_patterns_ohlc(data, patterns, title="Double Top Pattern Analysis", 
                                 volume=True, show_all_peaks=True, figsize=(16, 10)):
    """
    Plot OHLC chart with detected double top patterns using mplfinance
    
    Args:
        data (pd.DataFrame): DataFrame with OHLC data
        patterns (list): List of double top patterns from detect_double_top
        title (str): Chart title
        volume (bool): Whether to show volume subplot
        show_all_peaks (bool): Whether to show all detected peaks or only pattern peaks
        figsize (tuple): Figure size
    """
    addplots = []
    
    # Show all peaks if requested (as reference points)
    if show_all_peaks:
        maxima_idx = argrelextrema(data["Close"].values, np.greater, order=20)[0]
        if len(maxima_idx) > 0:
            all_peaks = pd.Series([np.nan] * len(data), index=data.index)
            all_peaks.iloc[maxima_idx] = data["Close"].iloc[maxima_idx]
            addplots.append(
                mpf.make_addplot(all_peaks, type='scatter', markersize=60, 
                               marker='v', color='gray', alpha=0.5)
            )
    
    # Plot each double top pattern
    colors = ['red', 'blue', 'orange', 'purple', 'brown', 'pink']
    
    for i, pattern in enumerate(patterns):
        color = colors[i % len(colors)]
        
        peak1_idx = pattern['peak1_idx']
        peak2_idx = pattern['peak2_idx']
        
        # Create scatter plots for the two peaks
        pattern_peaks = pd.Series([np.nan] * len(data), index=data.index)
        pattern_peaks.iloc[peak1_idx] = data["Close"].iloc[peak1_idx]
        pattern_peaks.iloc[peak2_idx] = data["Close"].iloc[peak2_idx]
        
        addplots.append(
            mpf.make_addplot(pattern_peaks, type='scatter', markersize=150, 
                           marker='v', color=color, alpha=0.9)
        )
        
        # Create connecting line between peaks
        peak1_price = data["Close"].iloc[peak1_idx]
        peak2_price = data["Close"].iloc[peak2_idx]
        
        # Create a line series between the two peaks
        peak_line = pd.Series([np.nan] * len(data), index=data.index)
        
        # Fill in values between the peaks to create a connecting line
        start_idx = min(peak1_idx, peak2_idx)
        end_idx = max(peak1_idx, peak2_idx)
        
        # Linear interpolation between peaks
        for idx in range(start_idx, end_idx + 1):
            progress = (idx - start_idx) / (end_idx - start_idx) if end_idx > start_idx else 0
            peak_line.iloc[idx] = peak1_price + (peak2_price - peak1_price) * progress
        
        addplots.append(
            mpf.make_addplot(peak_line, color=color, linestyle='--', 
                           width=2, alpha=0.7)
        )
        
        # Add neckline (horizontal line)
        neckline_price = pattern['neckline']
        neckline_series = pd.Series([neckline_price] * len(data), index=data.index)
        
        addplots.append(
            mpf.make_addplot(neckline_series, color=color, linestyle='-', 
                           width=3, alpha=0.8)
        )
        
        # Mark breakdown point if available
        if 'breakdown_idx' in pattern:
            breakdown_idx = pattern['breakdown_idx']
            breakdown_point = pd.Series([np.nan] * len(data), index=data.index)
            breakdown_point.iloc[breakdown_idx] = data["Close"].iloc[breakdown_idx]
            
            addplots.append(
                mpf.make_addplot(breakdown_point, type='scatter', markersize=200, 
                               marker='x', color=color, alpha=0.9)
            )
    
    # Create the chart with addplots
    fig, axes = mpf.plot(
        data,
        type='candle',
        style='charles',
        volume=volume,
        addplot=addplots if addplots else None,
        figsize=figsize,
        title=title,
        returnfig=True,
        tight_layout=True
    )
    
    # Create custom legend
    if patterns:
        legend_elements = []
        
        # Add pattern legend entries
        for i, pattern in enumerate(patterns):
            color = colors[i % len(colors)]
            peak1_price = data["Close"].iloc[pattern['peak1_idx']]
            peak2_price = data["Close"].iloc[pattern['peak2_idx']]
            neckline_price = pattern['neckline']
            
            legend_elements.append(
                plt.Line2D([0], [0], marker='v', color=color, linestyle='', 
                          markersize=10, 
                          label=f'Double Top {i+1}: Peaks ${peak1_price:.2f}, ${peak2_price:.2f}', 
                          alpha=0.9)
            )
            legend_elements.append(
                plt.Line2D([0], [0], color=color, linestyle='-', 
                          label=f'Neckline {i+1}: ${neckline_price:.2f}', 
                          linewidth=3, alpha=0.8)
            )
            
            if 'breakdown_idx' in pattern:
                breakdown_price = data["Close"].iloc[pattern['breakdown_idx']]
                legend_elements.append(
                    plt.Line2D([0], [0], marker='x', color=color, linestyle='', 
                              markersize=12, 
                              label=f'Breakdown {i+1}: ${breakdown_price:.2f}', 
                              alpha=0.9)
                )
        
        # Add reference peaks legend
        if show_all_peaks:
            legend_elements.append(
                plt.Line2D([0], [0], marker='v', color='gray', linestyle='', 
                          markersize=8, label='All Peaks', alpha=0.5)
            )
        
        # Add legend to the main price chart (first axis)
        axes[0].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1))
    
    plt.tight_layout()
    return fig, axes


def demo_double_top_ohlc(data, patterns):
    """Demo function to show double top patterns on OHLC chart"""
    print("Creating OHLC chart with double top patterns...")
    fig, axes = plot_double_top_patterns_ohlc(data, patterns, 
                                            title="S&P 500 Double Top Analysis")
    plt.show()
    return fig, axes

In [ ]:
def analyze_double_top_patterns(data, window=20, threshold=0.02):
    """
    Complete analysis function that detects and visualizes double top patterns
    :param data: DataFrame with OHLC price data
    :param window: Lookback window for finding peaks
    :param threshold: Price range to consider peaks as similar
    """
    print("Analyzing Double Top Patterns...")
    print(f"Parameters: window={window}, threshold={threshold*100}%")
    print("-" * 50)
    
    # Detect patterns
    patterns = detect_double_top(data, window=window, threshold=threshold)
    
    if patterns:
        print(f"Found {len(patterns)} double top pattern(s):")
        print()
        
        for i, pattern in enumerate(patterns, 1):
            peak1_idx = pattern['peak1_idx']
            peak2_idx = pattern['peak2_idx']
            peak1_price = data["Close"].iloc[peak1_idx]
            peak2_price = data["Close"].iloc[peak2_idx]
            
            # Handle date formatting safely
            try:
                if isinstance(data.index, pd.DatetimeIndex):
                    peak1_date = data.index[peak1_idx].strftime('%Y-%m-%d')
                    peak2_date = data.index[peak2_idx].strftime('%Y-%m-%d')
                else:
                    peak1_date = f"Index {peak1_idx}"
                    peak2_date = f"Index {peak2_idx}"
            except:
                peak1_date = f"Index {peak1_idx}"
                peak2_date = f"Index {peak2_idx}"
            
            price_diff = abs(peak1_price - peak2_price)
            price_diff_pct = (price_diff / peak1_price) * 100
            
            print(f"Pattern {i}:")
            print(f"  Peak 1: {peak1_date} at ${peak1_price:.2f}")
            print(f"  Peak 2: {peak2_date} at ${peak2_price:.2f}")
            print(f"  Price difference: ${price_diff:.2f} ({price_diff_pct:.2f}%)")
            print(f"  Neckline: ${pattern['neckline']:.2f}")
            
            if 'breakdown_idx' in pattern:
                breakdown_idx = pattern['breakdown_idx']
                breakdown_price = data["Close"].iloc[breakdown_idx]
                try:
                    if isinstance(data.index, pd.DatetimeIndex):
                        breakdown_date = data.index[breakdown_idx].strftime('%Y-%m-%d')
                    else:
                        breakdown_date = f"Index {breakdown_idx}"
                except:
                    breakdown_date = f"Index {breakdown_idx}"
                    
                print(f"  Breakdown: {breakdown_date} at ${breakdown_price:.2f}")
            
            # Calculate potential target (neckline - (peak - neckline))
            target = pattern['neckline'] - (peak1_price - pattern['neckline'])
            print(f"  Potential target: ${target:.2f}")
            print()
        
    else:
        print("No double top patterns detected with current parameters.")
        print("Try adjusting the window or threshold parameters:")
        print("- Increase window for broader peak detection")
        print("- Increase threshold to allow more price variation between peaks")
    
    return patterns

In [ ]:
# Example usage with comprehensive analysis using OHLC charts
try:
    print("=== DOUBLE TOP PATTERN ANALYSIS WITH OHLC CHARTS ===")
    print("Running Double Top Pattern Analysis...")
    patterns = analyze_double_top_patterns(df, window=15, threshold=0.03)
    
    if patterns:
        print("Displaying patterns on OHLC chart...")
        demo_double_top_ohlc(df, patterns)
    else:
        print("\nTrying with more relaxed parameters...")
        patterns = analyze_double_top_patterns(df, window=10, threshold=0.05)
        
        if patterns:
            print("Displaying patterns on OHLC chart...")
            demo_double_top_ohlc(df, patterns)
        else:
            print("No patterns found. Showing basic OHLC chart...")
            demo_basic_ohlc_chart(df)
        
except Exception as e:
    print(f"Error in double top OHLC analysis: {e}")
    print("Showing basic OHLC chart as fallback...")
    demo_basic_ohlc_chart(df)

## 4. False Breakout Detection
[top](#contents)

A **false breakout** (or "fakeout") occurs when price **temporarily breaks** through a support or resistance level but **quickly reverses**, failing to sustain the move:

### Pattern Types:

1. **False Resistance Breakout**:
   - Price breaks above resistance
   - Fails to sustain and reverses back down
   - Often traps breakout buyers

2. **False Support Breakout**:
   - Price breaks below support  
   - Fails to sustain and reverses back up
   - Often traps breakout sellers

### Visual Example:
```
Resistance -------------------- (false break above)
                              /\
                             /  \
Price                       /    \
                           /      \
           /\  /\  /\     /        \
          /  \/  \/  \   /          \
         /            \ /            \
Support ----------------\/------------- (holds)
```

### Algorithm Logic:

1. **Level Detection**: First identify support/resistance levels
2. **Breakout Detection**: Find when price crosses above/below levels
3. **Reversal Confirmation**: Check if price returns within specified timeframe
4. **Pattern Classification**: Categorize as false resistance or support breakout

### Key Parameters:
- **`lookback`**: Number of candles to check for reversal (default: 5)
- **`threshold`**: Price range to consider a level (default: 2%)

### Trading Applications:
- **Contrarian Signals**: False breakouts often lead to strong moves in opposite direction
- **Entry Opportunities**: Can provide low-risk entries when price returns to range
- **Market Psychology**: Reveals trapped traders and potential momentum shifts

In [ ]:
def detect_false_breakout(data, support_resistance, lookback=5, threshold=0.02):
    """
    Detect false breakouts above resistance or below support.
    :param data: DataFrame with 'Close' price column
    :param support_resistance: Dictionary from find_support_resistance
    :param lookback: Number of candles to check for reversal
    :param threshold: Price range to consider a level
    :return: List of false breakout events
    """
    false_breakouts = []

    # Check resistance breakouts
    for resistance in support_resistance["resistance"]:
        for i in range(len(data) - lookback):
            if (
                data["Close"].iloc[i] > resistance
                and data["Close"].iloc[i - 1] <= resistance
            ):
                # Breakout detected, check for reversal
                if any(
                    data["Close"].iloc[i + 1 : i + lookback + 1]
                    < resistance * (1 - threshold)
                ):
                    false_breakouts.append(
                        {
                            "type": "false_resistance_breakout",
                            "level": resistance,
                            "breakout_idx": i,
                        }
                    )

    # Check support breakouts
    for support in support_resistance["support"]:
        for i in range(len(data) - lookback):
            if data["Close"].iloc[i] < support and data["Close"].iloc[i - 1] >= support:
                # Breakout detected, check for reversal
                if any(
                    data["Close"].iloc[i + 1 : i + lookback + 1]
                    > support * (1 + threshold)
                ):
                    false_breakouts.append(
                        {
                            "type": "false_support_breakout",
                            "level": support,
                            "breakout_idx": i,
                        }
                    )

    return false_breakouts

In [ ]:
def plot_false_breakout_patterns_ohlc(data, false_breakouts, support_resistance, 
                                     title="False Breakout Analysis", volume=True, 
                                     lookback=5, figsize=(16, 10)):
    """
    Plot OHLC chart with detected false breakout patterns using mplfinance
    
    Args:
        data (pd.DataFrame): DataFrame with OHLC data
        false_breakouts (list): List of false breakout patterns from detect_false_breakout
        support_resistance (dict): Dictionary with support and resistance levels
        title (str): Chart title
        volume (bool): Whether to show volume subplot
        lookback (int): Number of candles used for reversal detection
        figsize (tuple): Figure size
    """
    addplots = []
    
    # Add support levels (green dashed lines)
    for i, level in enumerate(support_resistance["support"]):
        support_line = pd.Series([level] * len(data), index=data.index, name=f'Support_{i+1}')
        addplots.append(
            mpf.make_addplot(support_line, color='green', linestyle='--', width=2, alpha=0.6)
        )
    
    # Add resistance levels (red dashed lines)
    for i, level in enumerate(support_resistance["resistance"]):
        resistance_line = pd.Series([level] * len(data), index=data.index, name=f'Resistance_{i+1}')
        addplots.append(
            mpf.make_addplot(resistance_line, color='red', linestyle='--', width=2, alpha=0.6)
        )
    
    # Plot false breakout patterns
    colors = {'false_support_breakout': 'orange', 'false_resistance_breakout': 'purple'}
    markers = {'false_support_breakout': '^', 'false_resistance_breakout': 'v'}
    
    for i, breakout in enumerate(false_breakouts):
        breakout_type = breakout['type']
        level = breakout['level']
        breakout_idx = breakout['breakout_idx']
        color = colors[breakout_type]
        marker = markers[breakout_type]
        
        # Mark the breakout point
        breakout_point = pd.Series([np.nan] * len(data), index=data.index)
        breakout_point.iloc[breakout_idx] = data["Close"].iloc[breakout_idx]
        
        addplots.append(
            mpf.make_addplot(breakout_point, type='scatter', markersize=200, 
                           marker=marker, color=color, alpha=0.9)
        )
        
        # Highlight the reversal zone (next few candles after breakout)
        reversal_start = breakout_idx + 1
        reversal_end = min(breakout_idx + lookback + 1, len(data))
        
        if reversal_end > reversal_start:
            # Create series to highlight reversal candles
            reversal_zone = pd.Series([np.nan] * len(data), index=data.index)
            
            for idx in range(reversal_start, reversal_end):
                reversal_zone.iloc[idx] = data["Close"].iloc[idx]
            
            addplots.append(
                mpf.make_addplot(reversal_zone, type='scatter', markersize=80, 
                               marker='o', color=color, alpha=0.5)
            )
        
        # Draw vertical line from level to breakout point to show breakout distance
        level_line = pd.Series([np.nan] * len(data), index=data.index)
        level_line.iloc[breakout_idx] = level
        
        # Create a line showing the gap between level and breakout
        gap_line = pd.Series([np.nan] * len(data), index=data.index)
        breakout_price = data["Close"].iloc[breakout_idx]
        
        # Fill in a few points around the breakout to show the gap
        for offset in range(-2, 3):  # 5 points total
            idx = breakout_idx + offset
            if 0 <= idx < len(data):
                gap_line.iloc[idx] = level
        
        addplots.append(
            mpf.make_addplot(gap_line, color=color, linestyle=':', width=3, alpha=0.7)
        )
    
    # Create the chart with addplots
    fig, axes = mpf.plot(
        data,
        type='candle',
        style='charles',
        volume=volume,
        addplot=addplots if addplots else None,
        figsize=figsize,
        title=title,
        returnfig=True,
        tight_layout=True
    )
    
    # Create comprehensive custom legend
    legend_elements = []
    
    # Add support/resistance level legends
    if support_resistance["support"]:
        for i, level in enumerate(support_resistance["support"]):
            legend_elements.append(
                plt.Line2D([0], [0], color='green', linestyle='--', 
                          label=f'Support: ${level:.2f}', linewidth=2, alpha=0.6)
            )
    
    if support_resistance["resistance"]:
        for i, level in enumerate(support_resistance["resistance"]):
            legend_elements.append(
                plt.Line2D([0], [0], color='red', linestyle='--', 
                          label=f'Resistance: ${level:.2f}', linewidth=2, alpha=0.6)
            )
    
    # Add false breakout legends
    support_breakout_added = False
    resistance_breakout_added = False
    
    for breakout in false_breakouts:
        breakout_type = breakout['type']
        level = breakout['level']
        breakout_idx = breakout['breakout_idx']
        color = colors[breakout_type]
        marker = markers[breakout_type]
        breakout_price = data["Close"].iloc[breakout_idx]
        
        if breakout_type == 'false_support_breakout' and not support_breakout_added:
            legend_elements.append(
                plt.Line2D([0], [0], marker=marker, color=color, linestyle='', 
                          markersize=12, label='False Support Breakout', alpha=0.9)
            )
            legend_elements.append(
                plt.Line2D([0], [0], marker='o', color=color, linestyle='', 
                          markersize=8, label='Reversal Zone', alpha=0.5)
            )
            support_breakout_added = True
            
        elif breakout_type == 'false_resistance_breakout' and not resistance_breakout_added:
            legend_elements.append(
                plt.Line2D([0], [0], marker=marker, color=color, linestyle='', 
                          markersize=12, label='False Resistance Breakout', alpha=0.9)
            )
            legend_elements.append(
                plt.Line2D([0], [0], marker='o', color=color, linestyle='', 
                          markersize=8, label='Reversal Zone', alpha=0.5)
            )
            resistance_breakout_added = True
    
    # Add legend to the main price chart (first axis)
    if legend_elements:
        axes[0].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1))
    
    plt.tight_layout()
    return fig, axes


def demo_false_breakout_ohlc(data, false_breakouts, support_resistance, lookback=5):
    """Demo function to show false breakout patterns on OHLC chart"""
    print("Creating OHLC chart with false breakout patterns...")
    fig, axes = plot_false_breakout_patterns_ohlc(data, false_breakouts, support_resistance,
                                                 title="S&P 500 False Breakout Analysis",
                                                 lookback=lookback)
    plt.show()
    return fig, axes

In [ ]:
def analyze_false_breakout_patterns(data, window=20, min_touches=3, sr_threshold=0.02, 
                                  lookback=5, fb_threshold=0.02):
    """
    Complete analysis function that detects and visualizes false breakout patterns
    :param data: DataFrame with OHLC price data
    :param window: Lookback window for finding support/resistance levels
    :param min_touches: Minimum touches to confirm support/resistance
    :param sr_threshold: Threshold for support/resistance level detection
    :param lookback: Number of candles to check for breakout reversal
    :param fb_threshold: Threshold for false breakout detection
    """
    print("Analyzing False Breakout Patterns...")
    print(f"Parameters:")
    print(f"  S/R Detection: window={window}, min_touches={min_touches}, threshold={sr_threshold*100}%")
    print(f"  Breakout Detection: lookback={lookback}, threshold={fb_threshold*100}%")
    print("-" * 60)
    
    # First detect support and resistance levels
    support_resistance = find_support_resistance(data, window=window, 
                                                min_touches=min_touches, 
                                                threshold=sr_threshold)
    
    print(f"Support Levels Found: {len(support_resistance['support'])}")
    for i, level in enumerate(support_resistance['support'], 1):
        print(f"  {i}. ${level:.2f}")
    
    print(f"Resistance Levels Found: {len(support_resistance['resistance'])}")
    for i, level in enumerate(support_resistance['resistance'], 1):
        print(f"  {i}. ${level:.2f}")
    print()
    
    # Detect false breakouts
    false_breakouts = detect_false_breakout(data, support_resistance, 
                                          lookback=lookback, threshold=fb_threshold)
    
    if false_breakouts:
        print(f"Found {len(false_breakouts)} false breakout(s):")
        print()
        
        support_count = 0
        resistance_count = 0
        
        for breakout in false_breakouts:
            breakout_type = breakout['type']
            level = breakout['level']
            breakout_idx = breakout['breakout_idx']
            
            if breakout_type == 'false_support_breakout':
                support_count += 1
                pattern_num = support_count
                level_type = "Support"
            else:
                resistance_count += 1
                pattern_num = resistance_count
                level_type = "Resistance"
            
            # Get breakout details
            breakout_price = data["Close"].iloc[breakout_idx]
            
            try:
                if isinstance(data.index, pd.DatetimeIndex):
                    breakout_date = data.index[breakout_idx].strftime('%Y-%m-%d')
                else:
                    breakout_date = f"Index {breakout_idx}"
            except:
                breakout_date = f"Index {breakout_idx}"
            
            print(f"False {level_type} Breakout #{pattern_num}:")
            print(f"  Level: ${level:.2f}")
            print(f"  Breakout Date: {breakout_date}")
            print(f"  Breakout Price: ${breakout_price:.2f}")
            
            # Calculate breakout distance
            breakout_distance = abs(breakout_price - level)
            breakout_pct = (breakout_distance / level) * 100
            print(f"  Breakout Distance: ${breakout_distance:.2f} ({breakout_pct:.2f}%)")
            
            # Show reversal information
            reversal_start = breakout_idx + 1
            reversal_end = min(breakout_idx + lookback + 1, len(data))
            
            if reversal_end > reversal_start:
                reversal_prices = data["Close"].iloc[reversal_start:reversal_end]
                if breakout_type == 'false_support_breakout':
                    # Look for price moving back above support
                    reversal_price = reversal_prices.max()
                    reversal_condition = f"moved back above ${level:.2f}"
                else:
                    # Look for price moving back below resistance
                    reversal_price = reversal_prices.min()
                    reversal_condition = f"moved back below ${level:.2f}"
                
                print(f"  Reversal: Price {reversal_condition}")
                print(f"  Max reversal price in {lookback} candles: ${reversal_price:.2f}")
            
            print()
        
        print(f"Summary:")
        print(f"  False Support Breakouts: {support_count}")
        print(f"  False Resistance Breakouts: {resistance_count}")
        print()
        
    else:
        print("No false breakout patterns detected with current parameters.")
        print("Try adjusting the parameters:")
        print("- Decrease lookback for quicker reversal detection")
        print("- Increase fb_threshold for more sensitive breakout detection")
        print("- Adjust S/R parameters to find different levels")
    
    return false_breakouts, support_resistance

In [ ]:
# Example usage with comprehensive analysis using OHLC charts
try:
    print("=== FALSE BREAKOUT PATTERN ANALYSIS WITH OHLC CHARTS ===")
    print("Running False Breakout Pattern Analysis...")
    
    # Run analysis with default parameters
    false_breakouts, sr_levels = analyze_false_breakout_patterns(
        df, 
        window=20, 
        min_touches=2,  # Relaxed to find more levels
        sr_threshold=0.03,  # 3% threshold for S/R levels
        lookback=5, 
        fb_threshold=0.02  # 2% threshold for breakout detection
    )
    
    if false_breakouts:
        print("Displaying false breakout patterns on OHLC chart...")
        demo_false_breakout_ohlc(df, false_breakouts, sr_levels, lookback=5)
    else:
        # If no patterns found, try with more sensitive parameters
        print("\n" + "="*60)
        print("No patterns found with default parameters. Trying more sensitive settings...")
        print("="*60)
        
        false_breakouts, sr_levels = analyze_false_breakout_patterns(
            df,
            window=15,
            min_touches=2,
            sr_threshold=0.05,  # More relaxed S/R detection
            lookback=7,  # Longer reversal window
            fb_threshold=0.015  # More sensitive breakout detection
        )
        
        if false_breakouts:
            print("Displaying false breakout patterns on OHLC chart...")
            demo_false_breakout_ohlc(df, false_breakouts, sr_levels, lookback=7)
        else:
            print("Still no patterns found. Showing support/resistance levels on OHLC chart...")
            demo_support_resistance_ohlc(df, sr_levels)
        
except Exception as e:
    print(f"Error in false breakout OHLC analysis: {e}")
    print("Showing basic OHLC chart as fallback...")
    demo_basic_ohlc_chart(df)